In [1]:
import random
from pathlib import Path

import numpy as np
import optuna
from seabirdscientific.processing import MinVelocityType, loop_edit_pressure

from ctdam.conv import decode_hex
from ctdam.parser import CnvFile
from ctdam.proc.modules.seabird_functions import LoopRemoval

/home/lilith/PycharmProjects/ctdam/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# load data

In [2]:
data_dir = Path("../../../lr_dataset/").resolve()
out_dir = data_dir.parent / "cnv_out"
out_dir.mkdir(exist_ok=True)

for hex_path in data_dir.glob("**/*.hex"):
    out_path = out_dir / f"{hex_path.stem}.cnv"
    if out_path.exists():
        continue

    try:
        ctd = decode_hex(hex_path)
    except Exception:
        continue

    ctd.to_cnv(out_path)

/home/lilith/PycharmProjects/ctdam/src/ctdam/conv/cast_borders.py:138: RuntimeWarning: Found cast borders below the minimum cast size threshold of 634.51, defaulting to full cast size.
  warnings.warn(
/home/lilith/PycharmProjects/ctdam/.venv/lib/python3.14/site-packages/seabirdscientific/conversion.py:95: RuntimeWarning: divide by zero encountered in divide
  fLog = np.log(coefs.f0 / frequency)
/home/lilith/PycharmProjects/ctdam/.venv/lib/python3.14/site-packages/seabirdscientific/conversion.py:586: RuntimeWarning: divide by zero encountered in divide
  ts = np.log((KELVIN_OFFSET_25C - temperature) / (KELVIN_OFFSET_0C + temperature))
/home/lilith/PycharmProjects/ctdam/.venv/lib/python3.14/site-packages/seabirdscientific/conversion.py:587: RuntimeWarning: invalid value encountered in add
  a_term = a0 + a1 * ts + a2 * ts**2 + a3 * ts**3 + a4 * ts**4 + a5 * ts**5
/home/lilith/PycharmProjects/ctdam/.venv/lib/python3.14/site-packages/seabirdscientific/conversion.py:601: RuntimeWarning: di

# process dataset

In [11]:
cnv_dir = Path("../../../cnv_out").resolve()


def load_datasets() -> list[dict]:
    datasets = []

    for path in sorted(cnv_dir.glob("*.cnv")):
        try:
            ctd = CnvFile(path).to_ctd_data()
            pressure = ctd["prDM"].data
            sample_interval = 1.0 / ctd.sample_rate
            datasets.append(
                {
                    "name": path.name,
                    "pressure": pressure,
                    "sample_interval": sample_interval,
                    "latitude": ctd["latitude"].data,
                }
            )
        except Exception as e:
            print(f"skipping {path.name}:{e}")

    return datasets


dataset = load_datasets()

In [12]:
def split_dataset(dataset, test_ratio, seed=0):
    rng = random.Random(seed)
    idx = list(range(len(dataset)))
    rng.shuffle(idx)
    n_test = int(round(len(dataset) * test_ratio))
    test_idx = set(idx[:n_test])
    train = [dataset[i] for i in idx if i not in test_idx]
    test = [dataset[i] for i in idx if i in test_idx]
    return train, test

In [13]:
train_set, test_set = split_dataset(dataset, test_ratio=0.3)

# Hyperparameter Optimisiation

In [14]:
def monotonicity_score(
    pressure: np.ndarray, flags: np.ndarray
) -> tuple[float, int]:
    remaining_mask = ~flags
    remaining = pressure[remaining_mask]
    n_flags = int(np.sum(flags))

    if len(remaining) < 2:
        return float("nan"), n_flags

    diffs = np.diff(remaining)
    score = float((diffs > 0).sum()) / len(diffs)
    return score, n_flags

In [15]:
dataset

[{'name': '001-01.cnv',
  'pressure': array([  35.5  ,   35.5  ,   35.508, ..., 3824.812, 3824.836, 3824.836],
        shape=(103212,)),
  'sample_interval': np.float64(0.041666666666666664),
  'latitude': array([36.11642, 36.11642, 36.11642, ..., 36.11646, 36.11646, 36.11646],
        shape=(103212,))},
 {'name': '003-01.cnv',
  'pressure': array([   9.182,    9.236,    9.182, ..., 1020.146, 1020.146, 1020.201],
        shape=(41785,)),
  'sample_interval': np.float64(0.041666666666666664),
  'latitude': array([37.837  , 37.837  , 37.837  , ..., 37.83702, 37.83702, 37.83702],
        shape=(41785,))},
 {'name': '004-01.cnv',
  'pressure': array([  4.796,   4.734,   4.734, ..., 865.273, 865.336, 865.273],
        shape=(30819,)),
  'sample_interval': np.float64(0.041666666666666664),
  'latitude': array([37.84622, 37.84622, 37.84622, ..., 37.8462 , 37.8462 , 37.8462 ],
        shape=(30819,))},
 {'name': '007-01.cnv',
  'pressure': array([  48.937,   48.875,   48.875, ..., 2367.104, 23

### Jens

In [16]:
import numpy as np
from sklearn.model_selection import KFold


def make_objective_cv(dataset, remover, min_len=10, weight=0.5, k=5, seed=0):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    def obj(trial):
        params = dict(
            precut_period=trial.suggest_int("precut_period", 2, 20),
            cut_period=trial.suggest_int("cut_period", 2, 60),
            mean_speed_percent=trial.suggest_int("mean_speed_percent", 1, 60),
            delay=trial.suggest_int("delay", 1, 10),  # seconds
            filter_order=trial.suggest_categorical(
                "filter_order", [2, 3, 4, 5, 6]
            ),
        )

        fold_scores = []

        for _, val_idx in kf.split(dataset):
            val_fold = [dataset[i] for i in val_idx]

            mono_scores = []
            flag_props = []

            # math stops mathing if i dont include dis
            for item in val_fold:
                pressure = item["pressure"]
                if len(pressure) < min_len:
                    continue

                flags = remover.jens_loop_removal(
                    pressure=pressure,
                    sample_interval=item["sample_interval"],
                    **params,
                )

                score, _ = monotonicity_score(pressure=pressure, flags=flags)
                if np.isnan(score):
                    return -1e9

                mono_scores.append(score)
                flag_props.append(float(np.mean(flags)))

            if not mono_scores:
                return -1e9

            mean_mono = float(np.mean(mono_scores))
            mean_flags_good = 1.0 - float(np.mean(flag_props))
            fold_scores.append(
                weight * mean_mono + (1.0 - weight) * mean_flags_good
            )

        return float(np.mean(fold_scores))

    return obj

In [17]:
def evaluate(dataset, remover, params):
    scores = []
    for item in dataset:
        pressure = item["pressure"]
        sample_interval = item["sample_interval"]
        try:
            flags = remover.jens_loop_removal(
                pressure=pressure, sample_interval=sample_interval, **params
            )
        except ValueError:
            continue
        score, _ = monotonicity_score(pressure=pressure, flags=flags)
        if np.isnan(score):
            return -1e9
        scores.append(score)
    return float(np.mean(scores))

In [18]:
remover = LoopRemoval()

study = optuna.create_study(direction="maximize")  # new study
study.optimize(
    make_objective_cv(train_set, remover, min_len=10, weight=0.5, k=5, seed=0),
    n_trials=300,
)

[I 2026-06-29 16:21:39,697] A new study created in memory with name: no-name-1072e91f-40f0-4728-8cf0-76943f95547b
/home/lilith/PycharmProjects/ctdam/src/ctdam/proc/modules/seabird_functions.py:144: UserWarning: LoopRemoval is still in an experimental state. Be cautious with the results.
  warnings.warn(
[I 2026-06-29 16:21:39,968] Trial 0 finished with value: 0.6604297232274204 and parameters: {'precut_period': 8, 'cut_period': 36, 'mean_speed_percent': 6, 'delay': 7, 'filter_order': 2}. Best is trial 0 with value: 0.6604297232274204.
[I 2026-06-29 16:21:40,232] Trial 1 finished with value: 0.6558525713034957 and parameters: {'precut_period': 14, 'cut_period': 60, 'mean_speed_percent': 15, 'delay': 4, 'filter_order': 2}. Best is trial 0 with value: 0.6604297232274204.
[I 2026-06-29 16:21:40,503] Trial 2 finished with value: 0.6549867258438119 and parameters: {'precut_period': 6, 'cut_period': 58, 'mean_speed_percent': 35, 'delay': 10, 'filter_order': 4}. Best is trial 0 with value: 0.6

In [19]:
best_params = study.best_params
test_score = evaluate(test_set, remover, best_params)
print("best_params:", best_params)
print("test_score:", test_score)

best_params: {'precut_period': 3, 'cut_period': 2, 'mean_speed_percent': 7, 'delay': 10, 'filter_order': 4}
test_score: 0.5904963573868468


### Seabird

In [31]:
def sample_ef_params(trial):
    return dict(
        window_size=trial.suggest_float("window_size", 1.0, 60.0),
        mean_speed_percent=trial.suggest_float(
            "mean_speed_percent", 1.0, 60.0
        ),
        min_velocity=trial.suggest_float("min_velocity", 0.0, 5.0),
        min_soak_depth=trial.suggest_float("min_soak_depth", 0.0, 10.0),
        max_soak_depth=trial.suggest_float("max_soak_depth", 0.0, 20.0),
        remove_surface_soak=trial.suggest_categorical(
            "remove_surface_soak", [True, False]
        ),
        use_deck_pressure_offset=trial.suggest_categorical(
            "use_deck_pressure_offset", [True, False]
        ),
        exclude_flags=trial.suggest_categorical(
            "exclude_flags", [True, False]
        ),
        min_velocity_type=trial.suggest_categorical(
            "min_velocity_type",
            [MinVelocityType.FIXED, MinVelocityType.PERCENT],
        ),
    )


def monotonicity_objective_for_dataset(
    dataset,
    ef_params,
):
    mono_scores = []
    flag_props = []

    for item in dataset:
        pressure = item["pressure"]
        sample_interval = item["sample_interval"]
        latitude = item["latitude"]
        flag = np.array(
            [0.0 for _ in range(len(pressure))], dtype=float
        )  # filler
        try:
            edited_pressure = loop_edit_pressure(
                pressure=pressure,
                latitude=latitude,
                flag=flag,
                sample_interval=sample_interval,
                min_velocity_type=ef_params["min_velocity_type"],
                min_velocity=ef_params["min_velocity"],
                window_size=ef_params["window_size"],
                mean_speed_percent=ef_params["mean_speed_percent"],
                remove_surface_soak=ef_params["remove_surface_soak"],
                min_soak_depth=ef_params["min_soak_depth"],
                max_soak_depth=ef_params["max_soak_depth"],
                use_deck_pressure_offset=ef_params["use_deck_pressure_offset"],
                exclude_flags=ef_params["exclude_flags"],
                flag_value=-9.99e-29,
            )
        except ValueError:
            continue

        score, _ = monotonicity_score(pressure=pressure, flags=edited_pressure)
        if np.isnan(score):
            return None

        mono_scores.append(score)
        flag_props.append(float(np.mean(edited_pressure)))

    if not mono_scores:
        return None

    mean_mono = float(np.mean(mono_scores))
    mean_flags_good = 1.0 - float(np.mean(flag_props))
    return mean_mono, mean_flags_good


def make_objective_cv_valonly_ef(
    dataset, k=5, seed=42, min_len=10, weight=0.5
):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    def obj(trial):
        ef_params = sample_ef_params(trial)

        if ef_params["max_soak_depth"] < ef_params["min_soak_depth"]:
            return -1e9

        fold_scores = []
        for _, val_idx in kf.split(dataset):
            val_fold = [dataset[i] for i in val_idx]
            # same as above
            val_fold = [x for x in val_fold if len(x["pressure"]) >= min_len]
            if not val_fold:
                return -1e9

            out = monotonicity_objective_for_dataset(val_fold, ef_params)
            if out is None:
                return -1e9

            mean_mono, mean_flags_good = out
            fold_scores.append(
                weight * mean_mono + (1.0 - weight) * mean_flags_good
            )

        return float(np.mean(fold_scores))

    return obj

In [32]:
study = optuna.create_study(direction="maximize")
study.optimize(
    make_objective_cv_valonly_ef(
        train_set, min_len=10, weight=0.5, k=5, seed=0
    ),
    n_trials=300,
)

[I 2026-06-29 16:28:28,369] A new study created in memory with name: no-name-86fb153f-fa08-4f3d-9250-e9af38b07ca8
/tmp/ipykernel_43662/2705202451.py:13: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains MinVelocityType.FIXED which is of type MinVelocityType.
  min_velocity_type=trial.suggest_categorical("min_velocity_type", [MinVelocityType.FIXED, MinVelocityType.PERCENT])
/tmp/ipykernel_43662/2705202451.py:13: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains MinVelocityType.PERCENT which is of type MinVelocityType.
  min_velocity_type=trial.suggest_categorical("min_velocity_type", [MinVelocityType.FIXED, MinVelocityType.PERCENT])
[I 2026-06-29 16:28:29,744] Trial 0 finished with value: 0.5003329244503507 and parameters: {'window_size': 52.6886530032978, 'mean_speed_percent': 33.770902924746906, 'min_velocit

In [39]:
def evaluate_sb(dataset, params):
    scores = []
    for item in dataset:
        pressure = item["pressure"]
        sample_interval = item["sample_interval"]
        latitude = item["latitude"]
        flag = np.array([0.0 for _ in range(len(pressure))], dtype=float)
        try:
            flags = loop_edit_pressure(
                pressure=pressure,
                sample_interval=sample_interval,
                latitude=latitude,
                flag=flag,
                **params,
            )
        except ValueError:
            continue
        score, _ = monotonicity_score(pressure=pressure, flags=flags)
        if np.isnan(score):
            return -1e9
        scores.append(score)
    return float(np.mean(scores))

In [40]:
best_params = study.best_params
test_score = evaluate_sb(test_set, best_params)
print("best_params:", best_params)
print("test_score:", test_score)

best_params: {'window_size': 1.0063738046205128, 'mean_speed_percent': 3.335757872888925, 'min_velocity': 1.1787390153451158, 'min_soak_depth': 0.9617540739403488, 'max_soak_depth': 17.51304145305964, 'remove_surface_soak': False, 'use_deck_pressure_offset': False, 'exclude_flags': True, 'min_velocity_type': <MinVelocityType.PERCENT: 1>}
test_score: 0.3251627016231746
